In [1]:
print("Test")

Test


In [2]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

In [3]:
import os
import random
import time
import ast
import json
import tqdm
import pandas as pd
import numpy as np
import estnltk
from estnltk.default_resolver import make_resolver
from estnltk.taggers import VabamorfAnalyzer

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

from typing import Any

import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer, pipeline

from datasets import Dataset
from transformers.pipelines.pt_utils import KeyDataset

e:\Git_projects\EstNLTK\simpletransformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from scripts.notebooks.NotebookFunctions import display_examples

In [5]:
"""
Google Gen AI setup for Estonian marked-word rewriting.

This cell configures the Gen AI client, reads the API key from the environment,
and prepares the model call for JSON output.
"""

load_dotenv(".env", verbose=True)

api_key = os.getenv("GOOGLE_API_KEY")
model = "gemini-3.1-flash-lite-preview"
client = genai.Client(api_key=api_key)

print(f"Using model: {model}")

Using model: gemini-3.1-flash-lite-preview


In [63]:
bert_mlm_candidates_filepath = (
    HOMONYMS_DIRS["processed"] / "homonyms_bert_mlm_candidate_details_100.jsonl"
)
with open(bert_mlm_candidates_filepath, "r", encoding="utf-8") as f:
    bert_mlm_candidates_data = [json.loads(line) for line in f]

bert_mlm_candidates_df = pd.DataFrame(bert_mlm_candidates_data)
display(bert_mlm_candidates_df.head(1))

llm_homonyms_predictions_filepath = (
    HOMONYMS_DIRS["processed"] / "homonyms_llm_mlm_predictions_v3_200.parquet"
)
llm_homonyms_predictions_df = pd.read_parquet(llm_homonyms_predictions_filepath)
display(llm_homonyms_predictions_df.head(1))

row_index  inflection_type  \
0          0                1   

                                                                                                                                                                                                              sentence  \
0  Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.   

     word word_span true_label predicted_best_label  \
0  võitja  [74, 80]       sg n                 sg n   

   predicted_best_label_score  \
0                     0.72215   

                                                                                                                                                                                                                                      predicted_top_labels  \
0  [{'label': 'sg n', 'score': 0.7221502446191154}, {'label': 'sg g', 'score': 0.18655089150706772}, {'label': 'pl n', 'score': 0.031669486401369795}, {'label': 'nud', 'score': 0.03149387186567765}, {'label': 'sg tr', 'score': 0.0014702172484248877}]   

                                                                                                                                                                                                    label_scores  \
0  {'sg n': 0.7221502446191154, 'sg g': 0.18655089150706772, 'nud': 0.03149387186567765, 'pl n': 0.031669486401369795, 'sg tr': 0.0014702172484248877, 'o': 0.00035745027707889676, 's': 0.00035488014691509306}   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

,id,candidates,pred_label,true_label,predicted_form_distribution,source_sentence,target_word,word_span
0,630,"[ilmselge, arusaadav, loomulik, mõistetav, aktsepteeritav, kindel, tõestatud, ilmne, uskumatu, tunduv]",sg n,sg n,"{'': 0.025, '?': None, 'adt': None, 'nud': None, 'pl ab': None, 'pl all': None, 'pl el': None, 'pl g': None, 'pl ill': None, 'pl in': None, 'pl kom': None, 'pl n': 0.025, 'pl p': None, 's': None, 'sg all': None, 'sg el': None, 'sg g': None, 'sg ill': None, 'sg in': None, 'sg kom': None, 'sg n': 0.9249999999999999, 'sg p': None, 'sg ter': None, 'sg tr': None, 'tud': 0.025}","Aga samas on ju täiesti selge, et kool peab olema koht, kus turvatunne oleks tagatud kõigile, nii õpilastele kui õpetajatele.",selge,"[24, 29]"


In [64]:
print("Keys in 'candidate_details' of the first row:")
print(bert_mlm_candidates_df["candidate_details"].iloc[0][0].keys())

# Extract tokens from candidate_details and replace the original column with a new one containing new details in a more structured format.
for index, row in bert_mlm_candidates_df.iterrows():
    candidate_details = row["candidate_details"]
    structured_details = []
    for candidate in candidate_details:
        structured_details.append(
            {
                "token": candidate["token"],
                "score": candidate["score"],
                "resolved_form": candidate["resolved_forms"],
            }
        )
    bert_mlm_candidates_df.at[index, "candidate_details"] = structured_details

# Rename columns for clarity
bert_mlm_candidates_df.rename(columns={"row_index": "index"}, inplace=True)

Keys in 'candidate_details' of the first row:
dict_keys(['rank', 'token', 'score', 'sequence', 'candidate_sentence', 'candidate_span', 'resolved_forms'])


In [65]:
display(bert_mlm_candidates_df.head(1))

index  inflection_type  \
0      0                1   

                                                                                                                                                                                                              sentence  \
0  Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.   

     word word_span true_label predicted_best_label  \
0  võitja  [74, 80]       sg n                 sg n   

   predicted_best_label_score  \
0                     0.72215   

                                                                                                                                                                                                                                      predicted_top_labels  \
0  [{'label': 'sg n', 'score': 0.7221502446191154}, {'label': 'sg g', 'score': 0.18655089150706772}, {'label': 'pl n', 'score': 0.031669486401369795}, {'label': 'nud', 'score': 0.03149387186567765}, {'label': 'sg tr', 'score': 0.0014702172484248877}]   

                                                                                                                                                                                                    label_scores  \
0  {'sg n': 0.7221502446191154, 'sg g': 0.18655089150706772, 'nud': 0.03149387186567765, 'pl n': 0.031669486401369795, 'sg tr': 0.0014702172484248877, 'o': 0.00035745027707889676, 's': 0.00035488014691509306}   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [66]:
display_examples(
    bert_mlm_candidates_df.iloc[9:10],
    num_examples=1,
    display_or_print="print",
    list_of_columns_to_show=[
        c for c in bert_mlm_candidates_df.columns if c != "candidate_details"
    ],
)
print("Candidate details for the example:")
for candidate in bert_mlm_candidates_df["candidate_details"].iloc[9]:
    print(candidate)

Example 1 (index 9):
  index: 9
  inflection_type: 1
  sentence: Rahvusvaheline diplomaatia hoidis ära kahe naabri vahelise sõja.
  word: diplomaatia
  word_span: [15, 26]
  true_label: sg n
  predicted_best_label: sg n
  predicted_best_label_score: 0.87688607966993
  predicted_top_labels: [{'label': 'sg n', 'score': 0.87688607966993}, {'label': 'pl n', 'score': 0.003051594365388155}, {'label': 'sg g', 'score': 0.0023830299032852054}]
  label_scores: {'sg n': 0.87688607966993, 'pl n': 0.003051594365388155, 'sg g': 0.0023830299032852054}
  source: infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
----------------------------------------
Candidate details for the example:
{'token': 'kohus', 'score': 0.24282342195510864, 'resolved_form': ['sg n']}
{'token': 'kogukond', 'score': 0.11585666984319687, 'resolved_form': ['sg n']}
{'token': 'õigus', 'score': 0.10598563402891159, 'resolved_form': ['sg n']}
{'token': 'Kohus', 'score': 0.07946448028087616, 'resolved_form': ['sg n']}

In [82]:
# Create a new dataset of BERT MLM candidates containing only the relevant rows of LLM dataset
bert_mlm_candidates_subset_df = bert_mlm_candidates_df[
    bert_mlm_candidates_df["index"].isin(llm_homonyms_predictions_df["id"])
].reset_index(drop=True)
print(
    f"Number of rows in the BERT MLM candidates subset: {len(bert_mlm_candidates_subset_df)}"
)
print(
    f"Number of rows in the LLM homonyms predictions dataset: {len(llm_homonyms_predictions_df)}"
)
display(bert_mlm_candidates_subset_df.head(1))

Number of rows in the BERT MLM candidates subset: 200
Number of rows in the LLM homonyms predictions dataset: 200


index  inflection_type  \
0    160                1   

                                                             sentence  \
0  Harju politseiprefektuur algatas juhtunu uurimiseks kriminaalasja.   

      word word_span true_label predicted_best_label  \
0  juhtunu  [33, 40]       sg g                 sg g   

   predicted_best_label_score  \
0                    0.700327   

                                                                                                                                                                                                                                        predicted_top_labels  \
0  [{'label': 'sg g', 'score': 0.7003272122237831}, {'label': 'pl g', 'score': 0.20161150055355392}, {'label': 'sg n', 'score': 0.02562290756031871}, {'label': 'sg in', 'score': 0.0016581377130933106}, {'label': 'pl n', 'score': 0.0006457525305449963}]   

                                                                                                                                                                                                                                                                       label_scores  \
0  {'sg g': 0.7003272122237831, 'pl g': 0.20161150055355392, 'sg n': 0.02562290756031871, 'pl n': 0.0006457525305449963, 'tud': 0.0006457525305449963, 'sg in': 0.0016581377130933106, 'sg ad': 0.0006257191416807473, 'sg tr': 0.0004245820455253124, '?': 0.00036423635901883245}   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

### Testing on a single sentence


In [68]:
# System prompt: enforce Estonian rewriting behaviour, case conditioning, and JSON output
system_prompt = """You are an Estonian sentence rewriting assistant.
Your task is to select the best 5 candidates from a provided list of 100 BERT-generated candidate replacements for exactly one marked token in angle brackets <...>, using the full sentence context.

Follow this procedure internally:

1. Analyse the sentence context and the marked word.
2. Evaluate the 100 provided candidates only.
3. Reject candidates that do not fit the sentence grammatically or semantically.
4. Keep only candidates that are natural Estonian word forms in the correct context.
5. Rank the surviving candidates by suitability for the sentence.

Rules:
- Use only the provided BERT candidates. Do not invent new candidates.
- Preserve tense, number, case, agreement, punctuation, and capitalisation.
- The chosen candidates must fit the sentence context naturally and grammatically.
- The chosen candidates must be in one of the allowed morphological cases only: nominative, genitive, partitive, or illative.
- Prefer candidates whose morphological form best matches the masked word’s role in the sentence.
- If the marked token is a proper name, keep only proper-name candidates that fit the same grammatical context.
- If fewer than 5 valid candidates exist, return as many valid candidates as possible.
- Order the returned candidates from best to worst.
- Do not repeat the original marked word unless it is the only valid option.
- Return strict JSON only. Do not add explanations, markdown, or extra text.

JSON schema:
{
  "id": string,
  "candidates": array of 5 strings
}
"""

# Few-shot example to anchor the output format
few_shot_user = """Sentence ID: 2
Sentence: "Ehk oleks mõttekas ka mõni selleteemaline hoiatav <kampaania> korraldada," lisab punase autoga preili.
Candidates: ['aktsioon', 'üritus', 'kampaania', 'sündmus', 'saade', 'koosolek', 'kõne', 'päev', 'postitus', 'artikkel', 'video', 'näitus', 'asi', 'lugu', 'teade', 'lugemine', 'leht', 'liiklus', 'hoiatus', 'uudis', 'post', 'teavitus', 'loeng', 'komm', 'akt', 'õnnetus', 'sari', 'teema', 'tund', 'ettevõtmine', 'plakat', 'lõik', 'film', 'kirjutis', 'kirjandus', 'tekst', 'kontsert', 'seminar', 'märk', 'sõnum', 'konverents', 'üleskutse', 'foorum', 'festival', 'paraad', 'tegevus', 'sõit', 'reklaam', 'vms', 'eksperiment', 'paik', 'foto', 'kiri', 'nädal', 'ralli', 'juhtum', 'pilt', 'õppetund', 'ring', 'raamat', 'lehekülg', 'marss', 'jalutuskäik', 'pidu', 'koht', 'liik', 'küsitlus', 'mäng', 'traditsioon', 'kirjand', 'päeva', 'foor', 'avarii', 'õhtu', 'liikumine', 'kohtumine', 'arutelu', 'koolitus', 'võistlus', 'nali', 'liiklusõnnetus', 'ilm', 'tseremoonia', 'ajaleht', 'väljaanne', 'reis', 'näit', 'tee', 'materjal', 'test', 'püha', 'minut', 'kommentaarium', 'värk', 'näidis', 'katse', 'algatus', 'seeria', 'pühapäev', 'blogi']
"""

few_shot_assistant = json.dumps(
    {
        "id": "2",
        "candidates": [
            "aktsioon",
            "üritus",
            "sündmus",
            "näitus",
            "seminar",
        ],
    },
    ensure_ascii=False,
    indent=2,
)

In [38]:
# Real input sentence
user_prompt = """Sentence ID: 9
Sentence: Rahvusvaheline <diplomaatia> hoidis ära kahe naabri vahelise sõja.
Candidates: ['kohus', 'kogukond', 'õigus', 'Kohus', 'rahu', 'õiglus', 'sõda', 'koostöö', 'valitsus', 'Rahu', 'abi', 'kriis', 'komisjon', 'kokkulepe', 'Olümpiakomitee', 'olukord', 'poliitika', 'organisatsioon', 'õnnetus', 'tegevus', 'julgeolek', 'politsei', 'ja', 'arvamus', 'sündmus', 'jõud', 'maailm', 'katastroof', 'tava', 'Rist', 'otsus', 'Sõda', 'vandenõu', 'tegutsemine', 'kuritegu', 'Liit', 'Politsei', 'kohustus', 'rikkus', 'Venemaa', 'menetlus', 'konverents', 'uurimine', 'kohtunik', 'ühendus', 'majandus', 'majanduskriis', 'Jõud', 'kuritegevus', 'Kriminaal', 'sekkumine', 'lahendus', 'käitumine', 'konflikt', 'koalitsioon', 'Õigus', 'Föderatsioon', 'Vabariik', 'kindlustus', 'tragöödia', 'võim', 'seisukord', 'sõprus', 'avalikkus', 'vaen', 'Komisjon', 'huvi', 'kohtumine', 'revolutsioon', 'riik', 'Maailm', 'kaubandus', 'seisukoht', 'toetus', 'kord', 'vastutus', 'kokkuleppe', 'turvalisus', 'ajalugu', 'missioon', 'protsess', 'juhtum', 'seadus', 'asi', 'pääste', 'skandaal', 'kaitse', 'leping', 'Valitsus', 'liit', 'õiguse', 'Pank', 'elu', 'inimene', 'tulekahju', 'tõde', '–', 'operatsioon', 'praktika', 'maailmasõda']
"""

contents = [
    types.Content(role="user", parts=[types.Part.from_text(text=few_shot_user)]),
    types.Content(role="model", parts=[types.Part.from_text(text=few_shot_assistant)]),
    types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),
]


In [39]:
# Request a JSON response from Gemini / Google Gen AI
response = client.models.generate_content(
    model=model,
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        # temperature=0.3,
        top_p=0.95,
        # top_k=40,
        max_output_tokens=100000,
        response_mime_type="application/json",
    ),
)

INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"


In [40]:
content = response.text
result = json.loads(content)

print("Raw JSON response:")
print(json.dumps(result, ensure_ascii=False, indent=2))

Raw JSON response:
{
  "id": "9",
  "selected_candidates": [
    "koostöö",
    "sekkumine",
    "poliitika",
    "abi",
    "kaitse"
  ]
}


In [27]:
# Extract the selected candidates from the response
selected_id = "9"
selected_candidates = ["koostöö", "sekkumine", "poliitika", "abi", "kaitse"]

# Extract the selected candidates from the dataframe for comparison
original_candidates = bert_mlm_candidates_df.loc[int(selected_id), "candidate_details"]
selected_original_candidates = [
    c for c in original_candidates if c["token"] in selected_candidates
]
display(selected_original_candidates)

# Normalise the scores of the selected candidates for better comparison
scores = [c["score"] for c in selected_original_candidates]
total_score = sum(scores)
for candidate in selected_original_candidates:
    candidate["normalised_score"] = (
        candidate["score"] / total_score if total_score > 0 else 0
    )

display(selected_original_candidates)

# Sum up the normalised scores of the selected candidates per label
predictions = {}
for candidate in selected_original_candidates:
    label = candidate["resolved_form"][0]
    normalised_score = candidate["normalised_score"]
    predictions[label] = predictions.get(label, 0) + normalised_score
display(predictions)

# Print the final predicted label based on the highest summed normalised score
predicted_label = max(predictions, key=predictions.get)
print(f"Predicted label for sentence ID {selected_id}: {predicted_label}")

[{'token': 'koostöö',
  'score': 0.020630070939660072,
  'resolved_form': ['sg n'],
  'normalised_score': 0.5119477295561863},
 {'token': 'abi',
  'score': 0.009237299673259258,
  'resolved_form': ['sg n'],
  'normalised_score': 0.22922919697110355},
 {'token': 'poliitika',
  'score': 0.007941859774291515,
  'resolved_form': ['sg n'],
  'normalised_score': 0.19708206975119305},
 {'token': 'sekkumine',
  'score': 0.0016665436560288072,
  'resolved_form': ['sg n'],
  'normalised_score': 0.04135629215263725},
 {'token': 'kaitse',
  'score': 0.0008214472327381372,
  'resolved_form': ['sg n'],
  'normalised_score': 0.020384711568879887}]

[{'token': 'koostöö',
  'score': 0.020630070939660072,
  'resolved_form': ['sg n'],
  'normalised_score': 0.5119477295561863},
 {'token': 'abi',
  'score': 0.009237299673259258,
  'resolved_form': ['sg n'],
  'normalised_score': 0.22922919697110355},
 {'token': 'poliitika',
  'score': 0.007941859774291515,
  'resolved_form': ['sg n'],
  'normalised_score': 0.19708206975119305},
 {'token': 'sekkumine',
  'score': 0.0016665436560288072,
  'resolved_form': ['sg n'],
  'normalised_score': 0.04135629215263725},
 {'token': 'kaitse',
  'score': 0.0008214472327381372,
  'resolved_form': ['sg n'],
  'normalised_score': 0.020384711568879887}]

{'sg n': 1.0}

Predicted label for sentence ID 9: sg n


### Testing on a sample of sentences with batch processing and error handling


In [69]:
# System prompt: enforce Estonian rewriting behaviour, case conditioning, and JSON output
system_prompt = """You are an Estonian sentence rewriting assistant.
Your task is to select the best 5 candidates from a provided list of 100 BERT-generated candidate replacements for exactly one marked token in angle brackets <...>, using the full sentence context.

Follow this procedure internally:

1. Analyse the sentence context and the marked word.
2. Evaluate the 100 provided candidates only.
3. Reject candidates that do not fit the sentence grammatically or semantically.
4. Keep only candidates that are natural Estonian word forms in the correct context.
5. Rank the surviving candidates by suitability for the sentence.

Rules:
- Use only the provided BERT candidates. Do not invent new candidates.
- Preserve tense, number, case, agreement, punctuation, and capitalisation.
- The chosen candidates must fit the sentence context naturally and grammatically.
- The chosen candidates must be in one of the allowed morphological cases only: nominative, genitive, partitive, or illative.
- Prefer candidates whose morphological form best matches the masked word’s role in the sentence.
- If the marked token is a proper name, keep only proper-name candidates that fit the same grammatical context.
- If fewer than 5 valid candidates exist, return as many valid candidates as possible.
- Order the returned candidates from best to worst.
- Do not repeat the original marked word unless it is the only valid option.
- Return strict JSON only. Do not add explanations, markdown, or extra text.

JSON schema:
{
  "id": string,
  "candidates": array of 5 strings
}
"""

# Few-shot example to anchor the output format
few_shot_user = """Sentence ID: 2
Sentence: "Ehk oleks mõttekas ka mõni selleteemaline hoiatav <kampaania> korraldada," lisab punase autoga preili.
Candidates: ['aktsioon', 'üritus', 'kampaania', 'sündmus', 'saade', 'koosolek', 'kõne', 'päev', 'postitus', 'artikkel', 'video', 'näitus', 'asi', 'lugu', 'teade', 'lugemine', 'leht', 'liiklus', 'hoiatus', 'uudis', 'post', 'teavitus', 'loeng', 'komm', 'akt', 'õnnetus', 'sari', 'teema', 'tund', 'ettevõtmine', 'plakat', 'lõik', 'film', 'kirjutis', 'kirjandus', 'tekst', 'kontsert', 'seminar', 'märk', 'sõnum', 'konverents', 'üleskutse', 'foorum', 'festival', 'paraad', 'tegevus', 'sõit', 'reklaam', 'vms', 'eksperiment', 'paik', 'foto', 'kiri', 'nädal', 'ralli', 'juhtum', 'pilt', 'õppetund', 'ring', 'raamat', 'lehekülg', 'marss', 'jalutuskäik', 'pidu', 'koht', 'liik', 'küsitlus', 'mäng', 'traditsioon', 'kirjand', 'päeva', 'foor', 'avarii', 'õhtu', 'liikumine', 'kohtumine', 'arutelu', 'koolitus', 'võistlus', 'nali', 'liiklusõnnetus', 'ilm', 'tseremoonia', 'ajaleht', 'väljaanne', 'reis', 'näit', 'tee', 'materjal', 'test', 'püha', 'minut', 'kommentaarium', 'värk', 'näidis', 'katse', 'algatus', 'seeria', 'pühapäev', 'blogi']
"""

few_shot_assistant = json.dumps(
    {
        "id": "2",
        "candidates": [
            "aktsioon",
            "üritus",
            "sündmus",
            "näitus",
            "seminar",
        ],
    },
    ensure_ascii=False,
    indent=2,
)

In [88]:
# Create a user prompt template for the real sentences, using the same format as the few-shot example
def create_user_prompt(sentence_id, sentence, word_span):
    # Mark the target word with angle brackets
    start, end = word_span
    marked_sentence = (
        sentence[:start] + "<" + sentence[start:end] + ">" + sentence[end:]
    )
    return f"""Sentence ID: {sentence_id}
Sentence: {marked_sentence}
"""


def append_result_to_json(file_path, record, id=None):
    """Append one record to a JSON array stored in file_path."""
    file_path.parent.mkdir(parents=True, exist_ok=True)
    # If the file exists and is non-empty, load the existing data; otherwise start with an empty list
    if file_path.exists() and file_path.stat().st_size > 0:
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                data = []
    else:
        data = []

    if not isinstance(data, list):
        data = []
    if id:
        # If ID is provided, check if a record with the same ID already exists and update it; otherwise append the new record
        existing_record = next((r for r in data if int(r.get("id")) == int(id)), None)
        if existing_record:
            existing_record.update(record)
            # Remove error field if it exists, since we have a successful rewrite now
            existing_record.pop("error", None)
        else:
            data.append(
                record
            )  # Append the new record if no existing record with the same ID is found
    else:
        # If no ID is provided, put the record at the end of the list
        data.append(record)
    # Write the updated list back to the file with pretty formatting
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def rewrite_sentence_with_genai(
    sentence_id, sentence, word_span, candidate_details, metadata
):
    """Rewrite one sentence by selecting candidates and deriving the predicted label from BERT candidate scores."""
    user_prompt = create_user_prompt(
        sentence_id=sentence_id, sentence=sentence, word_span=word_span
    )
    # The few-shot example is included in every call to ensure the model understands the expected JSON format and the task.
    local_contents = [
        types.Content(role="user", parts=[types.Part.from_text(text=few_shot_user)]),
        types.Content(
            role="model", parts=[types.Part.from_text(text=few_shot_assistant)]
        ),
        types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),
    ]
    # Request a JSON response from Gemini / Google Gen AI
    response = client.models.generate_content(
        model=model,
        contents=local_contents,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            # temperature=0.3,
            top_p=0.95,
            max_output_tokens=100000,
            response_mime_type="application/json",
        ),
    )
    # Parse the JSON response and enrich it with metadata for later analysis
    content = response.text or "{}"
    parsed = json.loads(content)

    selected_candidates = parsed.get("candidates", [])
    if not isinstance(selected_candidates, list):
        selected_candidates = []

    selected_candidate_details = [
        candidate
        for candidate in candidate_details
        if candidate.get("token") in selected_candidates
    ]

    selected_score_total = sum(
        candidate.get("score", 0.0) for candidate in selected_candidate_details
    )
    for candidate in selected_candidate_details:
        candidate_score = candidate.get("score", 0.0)
        candidate["normalised_score"] = (
            candidate_score / selected_score_total if selected_score_total > 0 else 0.0
        )

    predictions = {}
    for candidate in selected_candidate_details:
        resolved_form = candidate.get("resolved_form", [])
        if not resolved_form:
            continue
        label = resolved_form[0]
        predictions[label] = predictions.get(label, 0.0) + candidate["normalised_score"]

    predicted_label = max(predictions, key=predictions.get) if predictions else None

    parsed["candidate_details"] = selected_candidate_details
    parsed["predictions"] = predictions
    parsed["pred_label"] = predicted_label

    parsed["source_sentence"] = sentence
    parsed["target_word"] = metadata.get("target_word", "")
    if isinstance(word_span, list):
        parsed["word_span"] = word_span
    else:
        parsed["word_span"] = word_span.astype(
            int
        ).tolist()  # Convert numpy array to list for JSON serialization
    parsed["label"] = metadata.get("label", [])

    return parsed


def get_latest_processed_id(output_file):
    """Get the highest sentence ID that has already been processed and saved in the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        with open(output_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list) and len(data) > 0:
                return max(int(record["id"]) for record in data if "id" in record)
    return -1  # No valid records found, start from the beginning


def get_unprocessed_dataset(dataset, output_file):
    """Filter the dataset to include only records that have errors or have not been processed yet, based on the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        unprocessed_records = []
        processed_records = []
        # First, get the list of sentences that have errors in the output file
        with open(output_file, "r", encoding="utf-8") as f:
            processed_data = json.load(f)
            for record in processed_data:
                if "error" in record:
                    unprocessed_records.append(record["source_sentence"])
                else:
                    processed_records.append(record["source_sentence"])
        # Now filter the original dataset to include only records that are either unprocessed or have errors
        filtered_dataset = dataset[
            dataset["sentence"].isin(unprocessed_records)
            | ~dataset["sentence"].isin(processed_records)
        ]
        return filtered_dataset
    return dataset  # If no output file exists, return the entire dataset for processing


def is_transient_error(exc: Exception) -> bool:
    """Heuristically detect retryable errors such as rate limit and network hiccups."""
    message = str(exc).lower()
    transient_markers = [
        "429",
        "rate",
        "quota",
        "too many requests",
        "timeout",
        "timed out",
        "connection",
        "temporar",
        "unavailable",
        "503",
        "502",
        "504",
        "internal",
    ]
    return any(marker in message for marker in transient_markers)


def rewrite_with_retry(
    sentence_id, sentence, word_span, candidate_details, metadata, config
):
    """Call rewrite function with exponential backoff on transient errors."""
    for attempt in range(1, config["max_attempts"] + 1):
        try:
            return rewrite_sentence_with_genai(
                sentence_id=sentence_id,
                sentence=sentence,
                word_span=word_span,
                candidate_details=candidate_details,
                metadata=metadata,
            )
        except Exception as exc:
            # should_retry = is_transient_error(exc) and attempt < config["max_attempts"]
            should_retry = (
                attempt < config["max_attempts"]
            )  # Retry on all exceptions for robustness, but still limit the number of attempts
            if not should_retry:
                raise
            backoff = min(
                config["max_backoff_seconds"],
                config["initial_backoff_seconds"] * (2 ** (attempt - 1)),
            )
            sleep_s = backoff + random.uniform(0.0, config["jitter_seconds"])
            print(
                f"Retry {attempt}/{config['max_attempts'] - 1} for sentence_id={sentence_id} "
                f"after transient error: {exc}. Sleeping {sleep_s:.2f}s..."
            )
            time.sleep(sleep_s)

In [89]:
##### Create a small sample of sentences from homonym_df for testing the batch processing and error handling
max_rows = 200  # Change or remove this limit for larger runs
working_df = bert_mlm_candidates_subset_df

### Calculate the number of groups based on inflection type and form label
# First check the number of unique inflection types
num_inflection_types = working_df["inflection_type"].nunique()
# Find the number of unique form labels for each inflection type
form_labels_per_inflection = working_df.groupby("inflection_type")[
    "true_label"
].nunique()
print("Number of unique form labels per inflection type:")
print(form_labels_per_inflection)
print(
    f"Number of unique inflection types: {num_inflection_types}"
)  # 4 inflection types in total (1, 16, 17, 19)
print(
    f"Number of inflection types and form label groups: {form_labels_per_inflection.sum()}"  # 10 inflection type + form label groups in total
)
print()

### Extract n sentences from each inflection type (1, 16, 17, 19) and from each form label (sg n, sg g, sg p, adt) to have a balanced test set across both dimensions
if max_rows >= form_labels_per_inflection.sum():
    cols = [c for c in working_df.columns if c not in ["inflection_type", "true_label"]]
    sample_df = (
        working_df.groupby(["inflection_type", "true_label"])[cols]
        .apply(
            lambda group: group.sample(
                n=int(max_rows / form_labels_per_inflection.sum()),
                random_state=seed,
            )
        )
        .reset_index(level=[0, 1])  # bring group keys
        .reset_index(
            drop=True
        )  # reset the final index to be sequential and drop the old one
    )
    # Display inflection type + form label distribution in the sample to verify balance
    print("Inflection type and form label distribution in the sample:")
    print(sample_df.groupby(["inflection_type", "true_label"]).size())

# If the sample is too small to be balanced, just take the first n rows and warn about the imbalance.
else:
    print(
        f"Warning: max_rows={max_rows} is too small to have a balanced sample across inflection types and form labels. The sample will be imbalanced."
    )
    sample_df = working_df.head(max_rows)

# Check few sample sentences and their metadata
display(sample_df.head())

Number of unique form labels per inflection type:
inflection_type
1     2
16    2
17    3
19    3
Name: true_label, dtype: int64
Number of unique inflection types: 4
Number of inflection types and form label groups: 10

Inflection type and form label distribution in the sample:
inflection_type  true_label
1                sg g          20
                 sg n          20
16               sg g          20
                 sg n          20
17               sg g          20
                 sg n          20
                 sg p          20
19               adt           20
                 sg g          20
                 sg p          20
dtype: int64


inflection_type true_label  index  \
0                1       sg g    160   
1                1       sg g   4623   
2                1       sg g   4502   
3                1       sg g    200   
4                1       sg g    833   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      sentence  \
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           Harju politseiprefektuur algatas juhtunu uurimiseks kriminaalasja.   
1  Väitekirja struktuur Tulenevalt väitekirja ülesehitusest õpik-monograafiana on töö eesmärkide saavutamiseks 1) uuritud maksukorralduse ja tarbimismaksude teoreetilisi aluseid ; 2) käsitletud käibemaksu rakendamise ajalugu ning põhjusi, mis mõjutasid selle rakendamist Euroopa Liidus või rakendamise edasilükkamist Ameerika Ühendriikides ; 3) analüüsitud käibemaksu puhul kasutatavat terminoloogiat ja käibemaksu rakendamise praktilisi probleeme ning selgitatud nende võimalikke lehendusi autori poolt väljatöötatud praktiliste näidetega ; 4) analüüsitud toiduainete käibemaksumäära alandamise mõju leibkondade elujärjele võrrelduna tulumaksuvaba miinimumi tõstmisega kaasneva mõjuga.   
2                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       Teisipäeval, 10. veebruaril teatas Indoneesia esindajatekoja asespiiker Abdul Gafur, et riigi parlament toetab valitsuse ettepanekut rakendada Indoneesia valuuta, ruupia stabiliseerimiseks valuutakomitee põhimõtet.   
3                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      26. veebruar, esmaspäev Kell 18.15-18.35 kadus Tihniku 5 Maksimarketi eest helebeezh VAZ 21063 numbriga 903 ABL, 1985. aasta väljalase.   
4                                                                                                                                                                                                                                                                                                                       

In [90]:
output_file = (
    OUTPUT_DIR / f"genai_annotations_bert_mlm_{max_rows}.json"
)  # Output file for results

# gemini-3.1-flash-lite-preview has strict request limits, so we throttle and retry safely.
config = {
    # Pacing configuration to avoid hitting rate limits
    "base_delay_seconds": round(
        60 / 15, 2
    ),  # RPM is 15 for gemini-3.1-flash-lite-preview
    "jitter_seconds": 1.0,
    # Retry configuration for transient failures
    "max_attempts": 6,
    "initial_backoff_seconds": 2.0,
    "max_backoff_seconds": 60.0,
}

# Start from the next unprocessed sentence based on the output file contents

# latest_id = get_latest_processed_id(output_file)
# unprocessed_df = sample_df[sample_df.index > latest_id]
# print(
#     f"Starting processing from sentence ID > {latest_id + 1}. Total unprocessed sentences: {len(unprocessed_df)}"
# )
unprocessed_df = get_unprocessed_dataset(sample_df, output_file)
if len(unprocessed_df) == 0:
    print("No unprocessed sentences found. All done!")
else:
    print(f"Total unprocessed sentences to process: {len(unprocessed_df)}")
    print("Next sentence to process:")
    display(unprocessed_df.head(1))

Total unprocessed sentences to process: 200
Next sentence to process:


inflection_type true_label  index  \
0                1       sg g    160   

                                                             sentence  \
0  Harju politseiprefektuur algatas juhtunu uurimiseks kriminaalasja.   

      word word_span predicted_best_label  predicted_best_label_score  \
0  juhtunu  [33, 40]                 sg g                    0.700327   

                                                                                                                                                                                                                                        predicted_top_labels  \
0  [{'label': 'sg g', 'score': 0.7003272122237831}, {'label': 'pl g', 'score': 0.20161150055355392}, {'label': 'sg n', 'score': 0.02562290756031871}, {'label': 'sg in', 'score': 0.0016581377130933106}, {'label': 'pl n', 'score': 0.0006457525305449963}]   

                                                                                                                                                                                                                                                                       label_scores  \
0  {'sg g': 0.7003272122237831, 'pl g': 0.20161150055355392, 'sg n': 0.02562290756031871, 'pl n': 0.0006457525305449963, 'tud': 0.0006457525305449963, 'sg in': 0.0016581377130933106, 'sg ad': 0.0006257191416807473, 'sg tr': 0.0004245820455253124, '?': 0.00036423635901883245}   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [91]:
# Find processed records that are in the sample_df
if output_file.exists() and output_file.stat().st_size > 0:
    with open(output_file, "r", encoding="utf-8") as f:
        processed_data = json.load(f)
        processed_ids = [
            int(record["id"]) for record in processed_data if "id" in record
        ]
        processed_sentences = sample_df[sample_df["index"].isin(processed_ids)]
        print(
            f"Number of sentences in sample_df that have already been processed: {len(processed_sentences)}"
        )
        print("Sample of processed sentences:")
        display(sample_df[sample_df["index"].isin(processed_ids)].head(10))

# Now remove all the already processed sentences from the output file that are not in the sample_df
if output_file.exists() and output_file.stat().st_size > 0:
    with open(output_file, "r", encoding="utf-8") as f:
        processed_data = json.load(f)
        processed_ids = [
            int(record["id"]) for record in processed_data if "id" in record
        ]
        filtered_data = [
            record
            for record in processed_data
            if int(record.get("id", -1)) in sample_df["index"].values
        ]
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(filtered_data, f, ensure_ascii=False, indent=2)

else:
    print("No output file found, so no sentences have been processed yet.")

No output file found, so no sentences have been processed yet.


In [92]:
# Batch rewrite loop: call Gen AI and append each result to a JSON file
processed = 0
for i, (sentence_id, sentence, word_span, word, label, candidate_details) in enumerate(
    zip(
        unprocessed_df["index"],
        unprocessed_df["sentence"],
        unprocessed_df["word_span"],
        unprocessed_df["word"],
        unprocessed_df["true_label"],
        unprocessed_df["candidate_details"],
    )
):
    try:
        metadata = {
            "target_word": word,
            "label": label.tolist() if isinstance(label, np.ndarray) else label,
        }
        result = rewrite_with_retry(
            sentence_id=sentence_id,
            sentence=sentence,
            word_span=word_span,
            candidate_details=candidate_details,
            config=config,
            metadata=metadata,
        )
        append_result_to_json(output_file, result, id=sentence_id)
        print(f"[{processed + 1}] Saved sentence_id={sentence_id}")
    except Exception as exc:
        error_record = {
            "id": str(sentence_id),
            "source_sentence": sentence,
            "target_word": word,
            "word_span": word_span.astype(int).tolist(),
            "true_label": label.tolist(),
            "pred_label": None,
            "error": str(exc),
        }
        append_result_to_json(output_file, error_record, id=sentence_id)
        print(f"[{processed + 1}] Error on sentence_id={sentence_id}: {exc}")

    processed += 1

    # Additional pacing between successful/failed items to avoid bursty traffic.
    if i < max_rows:
        sleep_s = config["base_delay_seconds"] + random.uniform(
            0.0, config["jitter_seconds"]
        )
        print(f"Sleeping {sleep_s:.2f}s before next request...")
        time.sleep(sleep_s)

print(f"Finished. Processed {processed} rows. Results appended to: {output_file}")

INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
[1] Saved sentence_id=160
Sleeping 4.75s before next request...
INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
[2] Saved sentence_id=4623
Sleeping 4.75s before next request...
INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
[3] Saved sentence_id=4502
Sleeping 4.79s before next request...
INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.goo

In [ ]:
# Open LLM sample annotation data
llm_samples_df = pd.read_json(output_file, orient="records")

new_records = []

# For each record, iterate over the candidates, construct the candidate sentence, and perform morphological analysis to extract the case and other features of the candidate replacement word. Assume each candidate has uniform probability and sum up the probabilities for candidates that match each form label to get a distribution of form labels for the candidates.
for index, row in tqdm.tqdm(llm_samples_df.iterrows(), total=llm_samples_df.shape[0]):
    id = row["id"]
    candidates = row.get("candidates", [])
    chosen = row.get("chosen", "")
    rewritten = row.get("rewritten", "")
    new_word_span = row.get("new_word_span", None)
    source_sentence = row["source_sentence"]
    target_word = row["target_word"]
    word_span = row["word_span"]
    label = row.get("label", [])[
        0
    ]  # Assuming label is a list and we take the first element as the form label
    # print(f"Source sentence: {source_sentence}")
    # print(f"Target word span: {word_span}")
    candidate_form_weight = (
        1 / len(candidates) if candidates else 0
    )  # Uniform weight for each candidate
    form_probabilities = {}
    for candidate in candidates:
        # Construct sentence with the candidate replacement
        candidate_sentence = (
            source_sentence[: word_span[0]]
            + candidate
            + source_sentence[word_span[1] :]
        )
        # print(f"Candidate sentence: {candidate_sentence}")
        # Create EstNLTK Text object and perform morphological analysis
        estnltk_text = estnltk.Text(candidate_sentence)
        estnltk_text.tag_layer("morph_analysis")
        # Extract the morphological tags for the candidate word
        morph_layer = estnltk_text.morph_analysis
        # Find the token in the morph layer that corresponds to the candidate replacement
        candidate_token = None
        for token in morph_layer:
            if (
                token.start <= word_span[0] < token.end
            ):  # Check if the token covers the start of the original word span
                candidate_token = token
                break
        if candidate_token:
            # Count up the form labels for the candidate
            number_of_labels = len(candidate_token.form) if candidate_token.form else 0
            if number_of_labels > 0:
                weight_per_label = candidate_form_weight / number_of_labels
                for candidate_label in candidate_token.form:
                    form_probabilities[candidate_label] = (
                        form_probabilities.get(candidate_label, 0) + weight_per_label
                    )
            else:
                form_probabilities["unknown"] = (
                    form_probabilities.get("unknown", 0) + candidate_form_weight
                )

    best_form_label = (
        max(form_probabilities, key=form_probabilities.get)
        if form_probabilities
        else "unknown"
    )
    # Create a record for the chosen candidate and its features
    candidate_record = {
        "id": id,
        "candidates": candidates,
        "pred_label": best_form_label,
        "true_label": label,
        "predicted_form_distribution": form_probabilities,
        "source_sentence": source_sentence,
        "target_word": target_word,
        "word_span": word_span,
    }
    new_records.append(candidate_record)

100%|██████████| 100/100 [00:06<00:00, 14.64it/s]


In [ ]:
# Create a new DataFrame from the predicted form labels and their distributions and save to JSON
homonym_results_df = pd.DataFrame(new_records)
homonym_results_df_path = (
    HOMONYMS_DIRS["processed"] / f"homonyms_llm_mlm_predictions_v2_{max_rows}.parquet"
)
homonym_results_df.to_parquet(homonym_results_df_path, index=False)